<a href="https://colab.research.google.com/github/rafzur1-commits/pythonstuff/blob/main/Joinstasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.executescript("""
CREATE TABLE salesman (
    salesman_id INT PRIMARY KEY,
    name TEXT,
    city TEXT,
    commission REAL
);

CREATE TABLE customer (
    customer_id INT PRIMARY KEY,
    cust_name TEXT,
    city TEXT,
    grade INT,
    salesman_id INT
);

CREATE TABLE orders (
    ord_no INT PRIMARY KEY,
    purch_amt REAL,
    ord_date TEXT,
    customer_id INT,
    salesman_id INT
);

INSERT INTO salesman VALUES
(5001,'James Hoog','New York',0.15),
(5002,'Nail Knite','Paris',0.13),
(5005,'Pit Alex','London',0.11),
(5006,'Mc Lyon','Paris',0.14),
(5007,'Paul Adam','Rome',0.13),
(5003,'Lauson Hen','San Jose',0.12);

INSERT INTO customer VALUES
(3002,'Nick Rimando','New York',100,5001),
(3007,'Brad Davis','New York',200,5001),
(3005,'Graham Zusi','California',200,5002),
(3008,'Julian Green','London',300,5002),
(3004,'Fabian Johnson','Paris',300,5006),
(3009,'Geoff Cameron','Berlin',100,5003),
(3003,'Jozy Altidor','Moscow',200,5007),
(3001,'Brad Guzan','London',NULL,5005);

INSERT INTO orders VALUES
(70001,150.5,'2012-10-05',3005,5002),
(70009,270.65,'2012-09-10',3001,5005),
(70002,65.26,'2012-10-05',3002,5001),
(70004,110.5,'2012-08-17',3009,5003),
(70007,948.5,'2012-09-10',3005,5002),
(70005,2400.6,'2012-07-27',3007,5001),
(70008,5760,'2012-09-10',3002,5001),
(70010,1983.43,'2012-10-10',3004,5006);
""")

In [ ]:
pd.read_sql_query("""
SELECT s.name AS Salesman, c.cust_name, c.city
FROM salesman s
JOIN customer c ON s.city = c.city;
""", conn) #this is to print out salesmen and customers who reside in the same city

,Salesman,cust_name,city
0,James Hoog,Brad Davis,New York
1,James Hoog,Nick Rimando,New York
2,Nail Knite,Fabian Johnson,Paris
3,Pit Alex,Brad Guzan,London
4,Pit Alex,Julian Green,London
5,Mc Lyon,Fabian Johnson,Paris


In [ ]:
pd.read_sql_query("""
SELECT o.ord_no, o.purch_amt, c.cust_name, c.city
FROM orders o
JOIN customer c ON o.customer_id = c.customer_id
WHERE o.purch_amt BETWEEN 500 AND 2000;
""", conn) #those orders who exist between 500 and 2000, plus customer ID

,ord_no,purch_amt,cust_name,city
0,70007,948.50,Graham Zusi,California
1,70010,1983.43,Fabian Johnson,Paris


In [ ]:
pd.read_sql_query("""
SELECT c.cust_name, c.city, s.name AS Salesman, s.commission
FROM customer c
JOIN salesman s ON c.salesman_id = s.salesman_id;
""", conn) #salesperson and assigned customer

,cust_name,city,Salesman,commission
0,Nick Rimando,New York,James Hoog,0.15
1,Brad Davis,New York,James Hoog,0.15
2,Graham Zusi,California,Nail Knite,0.13
3,Julian Green,London,Nail Knite,0.13
4,Fabian Johnson,Paris,Mc Lyon,0.14
5,Geoff Cameron,Berlin,Lauson Hen,0.12
6,Jozy Altidor,Moscow,Paul Adam,0.13
7,Brad Guzan,London,Pit Alex,0.11


In [ ]:
pd.read_sql_query("""
SELECT c.cust_name, c.city, s.name AS Salesman, s.commission
FROM customer c
JOIN salesman s ON c.salesman_id = s.salesman_id
WHERE s.commission > 0.12;
""", conn) #salespeople with more than 12% commission

,cust_name,city,Salesman,commission
0,Nick Rimando,New York,James Hoog,0.15
1,Brad Davis,New York,James Hoog,0.15
2,Graham Zusi,California,Nail Knite,0.13
3,Julian Green,London,Nail Knite,0.13
4,Fabian Johnson,Paris,Mc Lyon,0.14
5,Jozy Altidor,Moscow,Paul Adam,0.13


In [ ]:
pd.read_sql_query("""
SELECT c.cust_name, c.city AS customer_city,
       s.name AS Salesman, s.city AS salesman_city, s.commission
FROM customer c
JOIN salesman s ON c.salesman_id = s.salesman_id
WHERE c.city <> s.city AND s.commission > 0.12;
""", conn) #salespeeps with >12% commission but don't live in the same cities as their customers

,cust_name,customer_city,Salesman,salesman_city,commission
0,Graham Zusi,California,Nail Knite,Paris,0.13
1,Julian Green,London,Nail Knite,Paris,0.13
2,Jozy Altidor,Moscow,Paul Adam,Rome,0.13


In [ ]:
pd.read_sql_query("""
SELECT o.ord_no, o.ord_date, o.purch_amt,
       c.cust_name, c.grade,
       s.name AS Salesman, s.commission
FROM orders o
JOIN customer c ON o.customer_id = c.customer_id
JOIN salesman s ON o.salesman_id = s.salesman_id;
""", conn) #details of an order

,ord_no,ord_date,purch_amt,cust_name,grade,Salesman,commission
0,70001,2012-10-05,150.50,Graham Zusi,200.0,Nail Knite,0.13
1,70009,2012-09-10,270.65,Brad Guzan,NaN,Pit Alex,0.11
2,70002,2012-10-05,65.26,Nick Rimando,100.0,James Hoog,0.15
3,70004,2012-08-17,110.50,Geoff Cameron,100.0,Lauson Hen,0.12
4,70007,2012-09-10,948.50,Graham Zusi,200.0,Nail Knite,0.13
5,70005,2012-07-27,2400.60,Brad Davis,200.0,James Hoog,0.15
6,70008,2012-09-10,5760.00,Nick Rimando,100.0,James Hoog,0.15
7,70010,2012-10-10,1983.43,Fabian Johnson,300.0,Mc Lyon,0.14


In [ ]:
pd.read_sql_query("""
SELECT o.ord_no, o.purch_amt, o.ord_date,
       c.customer_id, c.cust_name, c.city, c.grade,
       s.salesman_id, s.name, s.city AS salesman_city, s.commission
FROM orders o
JOIN customer c ON o.customer_id = c.customer_id
JOIN salesman s ON o.salesman_id = s.salesman_id;
""", conn) #joining all the order, cust, and salesp tables together

,ord_no,purch_amt,ord_date,customer_id,cust_name,city,grade,salesman_id,name,salesman_city,commission
0,70001,150.50,2012-10-05,3005,Graham Zusi,California,200.0,5002,Nail Knite,Paris,0.13
1,70009,270.65,2012-09-10,3001,Brad Guzan,London,NaN,5005,Pit Alex,London,0.11
2,70002,65.26,2012-10-05,3002,Nick Rimando,New York,100.0,5001,James Hoog,New York,0.15
3,70004,110.50,2012-08-17,3009,Geoff Cameron,Berlin,100.0,5003,Lauson Hen,San Jose,0.12
4,70007,948.50,2012-09-10,3005,Graham Zusi,California,200.0,5002,Nail Knite,Paris,0.13
5,70005,2400.60,2012-07-27,3007,Brad Davis,New York,200.0,5001,James Hoog,New York,0.15
6,70008,5760.00,2012-09-10,3002,Nick Rimando,New York,100.0,5001,James Hoog,New York,0.15
7,70010,1983.43,2012-10-10,3004,Fabian Johnson,Paris,300.0,5006,Mc Lyon,Paris,0.14


In [ ]:
pd.read_sql_query("""
SELECT c.cust_name, c.city, c.grade,
       s.name AS Salesman, s.city AS salesman_city
FROM customer c
JOIN salesman s ON c.salesman_id = s.salesman_id
ORDER BY c.customer_id;
""", conn) #customer name, city, salesp and salesp city all ordered by customer id ascendingly

,cust_name,city,grade,Salesman,salesman_city
0,Brad Guzan,London,NaN,Pit Alex,London
1,Nick Rimando,New York,100.0,James Hoog,New York
2,Jozy Altidor,Moscow,200.0,Paul Adam,Rome
3,Fabian Johnson,Paris,300.0,Mc Lyon,Paris
4,Graham Zusi,California,200.0,Nail Knite,Paris
5,Brad Davis,New York,200.0,James Hoog,New York
6,Julian Green,London,300.0,Nail Knite,Paris
7,Geoff Cameron,Berlin,100.0,Lauson Hen,San Jose


In [ ]:
pd.read_sql_query("""
SELECT c.cust_name, c.city, c.grade,
       s.name AS Salesman, s.city AS salesman_city
FROM customer c
JOIN salesman s ON c.salesman_id = s.salesman_id
WHERE c.grade < 300
ORDER BY c.customer_id;
""", conn) #customers graded less than 300 and ordered ascendingly by customer id

,cust_name,city,grade,Salesman,salesman_city
0,Nick Rimando,New York,100,James Hoog,New York
1,Jozy Altidor,Moscow,200,Paul Adam,Rome
2,Graham Zusi,California,200,Nail Knite,Paris
3,Brad Davis,New York,200,James Hoog,New York
4,Geoff Cameron,Berlin,100,Lauson Hen,San Jose


In [ ]:
pd.read_sql_query("""
SELECT c.cust_name, c.city,
       o.ord_no, o.ord_date, o.purch_amt
FROM customer c
LEFT JOIN orders o ON c.customer_id = o.customer_id
ORDER BY o.ord_date;
""", conn)

,cust_name,city,ord_no,ord_date,purch_amt
0,Julian Green,London,NaN,None,NaN
1,Jozy Altidor,Moscow,NaN,None,NaN
2,Brad Davis,New York,70005.0,2012-07-27,2400.60
3,Geoff Cameron,Berlin,70004.0,2012-08-17,110.50
4,Nick Rimando,New York,70008.0,2012-09-10,5760.00
5,Graham Zusi,California,70007.0,2012-09-10,948.50
6,Brad Guzan,London,70009.0,2012-09-10,270.65
7,Nick Rimando,New York,70002.0,2012-10-05,65.26
8,Graham Zusi,California,70001.0,2012-10-05,150.50
9,Fabian Johnson,Paris,70010.0,2012-10-10,1983.43


In [ ]:
pd.read_sql_query("""
SELECT c.cust_name, c.city,
       o.ord_no, o.ord_date, o.purch_amt,
       s.name AS Salesman, s.commission
FROM customer c
LEFT JOIN orders o ON c.customer_id = o.customer_id
LEFT JOIN salesman s ON c.salesman_id = s.salesman_id;
""", conn)

,cust_name,city,ord_no,ord_date,purch_amt,Salesman,commission
0,Nick Rimando,New York,70002.0,2012-10-05,65.26,James Hoog,0.15
1,Nick Rimando,New York,70008.0,2012-09-10,5760.00,James Hoog,0.15
2,Brad Davis,New York,70005.0,2012-07-27,2400.60,James Hoog,0.15
3,Graham Zusi,California,70001.0,2012-10-05,150.50,Nail Knite,0.13
4,Graham Zusi,California,70007.0,2012-09-10,948.50,Nail Knite,0.13
5,Julian Green,London,NaN,None,NaN,Nail Knite,0.13
6,Fabian Johnson,Paris,70010.0,2012-10-10,1983.43,Mc Lyon,0.14
7,Geoff Cameron,Berlin,70004.0,2012-08-17,110.50,Lauson Hen,0.12
8,Jozy Altidor,Moscow,NaN,None,NaN,Paul Adam,0.13
9,Brad Guzan,London,70009.0,2012-09-10,270.65,Pit Alex,0.11


In [ ]:
pd.read_sql_query("""
SELECT DISTINCT s.salesman_id, s.name, s.city, s.commission
FROM salesman s
LEFT JOIN customer c ON s.salesman_id = c.salesman_id
ORDER BY s.salesman_id;
""", conn)

,salesman_id,name,city,commission
0,5001,James Hoog,New York,0.15
1,5002,Nail Knite,Paris,0.13
2,5003,Lauson Hen,San Jose,0.12
3,5005,Pit Alex,London,0.11
4,5006,Mc Lyon,Paris,0.14
5,5007,Paul Adam,Rome,0.13


In [ ]:
pd.read_sql_query("""
SELECT s.name AS Salesman, s.city AS salesman_city,
       c.cust_name, c.city AS customer_city, c.grade,
       o.ord_no, o.ord_date, o.purch_amt
FROM salesman s
LEFT JOIN customer c ON s.salesman_id = c.salesman_id
LEFT JOIN orders o ON c.customer_id = o.customer_id;
""", conn)

,Salesman,salesman_city,cust_name,customer_city,grade,ord_no,ord_date,purch_amt
0,James Hoog,New York,Nick Rimando,New York,100.0,70002.0,2012-10-05,65.26
1,James Hoog,New York,Nick Rimando,New York,100.0,70008.0,2012-09-10,5760.00
2,James Hoog,New York,Brad Davis,New York,200.0,70005.0,2012-07-27,2400.60
3,Nail Knite,Paris,Graham Zusi,California,200.0,70001.0,2012-10-05,150.50
4,Nail Knite,Paris,Graham Zusi,California,200.0,70007.0,2012-09-10,948.50
5,Nail Knite,Paris,Julian Green,London,300.0,NaN,None,NaN
6,Pit Alex,London,Brad Guzan,London,NaN,70009.0,2012-09-10,270.65
7,Mc Lyon,Paris,Fabian Johnson,Paris,300.0,70010.0,2012-10-10,1983.43
8,Paul Adam,Rome,Jozy Altidor,Moscow,200.0,NaN,None,NaN
9,Lauson Hen,San Jose,Geoff Cameron,Berlin,100.0,70004.0,2012-08-17,110.50


In [ ]:
pd.read_sql_query("""
SELECT DISTINCT s.salesman_id, s.name, s.city, s.commission
FROM salesman s
LEFT JOIN customer c ON s.salesman_id = c.salesman_id
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE (o.purch_amt >= 2000 AND c.grade IS NOT NULL)
   OR o.ord_no IS NULL;
""", conn)

,salesman_id,name,city,commission
0,5001,James Hoog,New York,0.15
1,5002,Nail Knite,Paris,0.13
2,5007,Paul Adam,Rome,0.13
